trains Linear Regression, Random Forest, and XGBoost on:

1. Non-PCA features
2. PCA features

In [13]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


## 1. Imports and paths

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")  # safe for notebook saving

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
    print("XGBoost imported successfully.")
except Exception as e:
    XGBOOST_AVAILABLE = False
    print("XGBoost import failed:", e)

PROJECT_DIR = Path("/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project")

NO_DR_DIR  = PROJECT_DIR / "model_ready_no_dr"
PCA_DIR    = PROJECT_DIR / "model_ready_with_pca"
OUTPUT_DIR = PROJECT_DIR / "model_outputs" / "baseline_models"
PLOT_DIR   = OUTPUT_DIR / "plots"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

target_col = "target_energy_next_60s_wh"
id_cols    = ["log_id", "dataset_label"]

print("Output directory:", OUTPUT_DIR)
print("Plot directory  :", PLOT_DIR)

XGBoost imported successfully.
Output directory: /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/baseline_models
Plot directory  : /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/baseline_models/plots


## 2. Helper functions

In [29]:
#metric helpers
def rmse(y_true, y_pred):
    '''
    The rmse() function was used to calculate Root Mean Squared Error. This helped measure the average prediction error, while giving more weight to larger errors.
    '''
    return np.sqrt(mean_squared_error(y_true, y_pred))


def get_feature_cols(df, target_col, id_cols):
    '''
    used to separate the input features from the target and identifier columns. Columns such as log_id, dataset_label, and the target variable were removed so that only model input variables were used for training.
    '''
    return [c for c in df.columns if c not in id_cols + [target_col]]


#core evaluation

def evaluate_model(model_name, dataset_name, model,
                   X_train, y_train,
                   X_val,   y_val,
                   X_test,  y_test,
                   test_df=None):
    """
    Fits model, returns a results dict with MAE / RMSE / R² for val + test,
    plus per-label test metrics and flight-level test MAE if test_df supplied.
    """
    print(f"\n{'='*60}")
    print(f"  Training : {model_name}  |  Dataset : {dataset_name}")
    print(f"{'='*60}")

    model.fit(X_train, y_train)

    val_pred  = model.predict(X_val)
    test_pred = model.predict(X_test)

    results = {
        "dataset"   : dataset_name,
        "model"     : model_name,
        # --- validation ---
        "val_MAE"   : mean_absolute_error(y_val, val_pred),
        "val_RMSE"  : rmse(y_val, val_pred),
        "val_R2"    : r2_score(y_val, val_pred),
        # --- test overall ---
        "test_MAE"  : mean_absolute_error(y_test, test_pred),
        "test_RMSE" : rmse(y_test, test_pred),
        "test_R2"   : r2_score(y_test, test_pred),
    }

    #per-label test breakdown
    if test_df is not None:
        for label in ["normal", "abnormal"]:
            mask = test_df["dataset_label"].values == label
            if mask.sum() == 0:
                continue
            yt = y_test.values[mask]
            yp = test_pred[mask]
            results[f"test_MAE_{label}"]  = mean_absolute_error(yt, yp)
            results[f"test_RMSE_{label}"] = rmse(yt, yp)
            results[f"test_R2_{label}"]   = r2_score(yt, yp)
            print(f"  [{label:8s}]  MAE={results[f'test_MAE_{label}']:.4f}  "
                  f"RMSE={results[f'test_RMSE_{label}']:.4f}  "
                  f"R²={results[f'test_R2_{label}']:.4f}")

    #flight-level MAE
    if test_df is not None and "log_id" in test_df.columns:
        tmp = test_df[["log_id"]].copy()
        tmp["y_true"] = y_test.values
        tmp["y_pred"] = test_pred
        flight_mae = (
            tmp.groupby("log_id")
               .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))
               .reset_index()
               .rename(columns={0: "flight_MAE"})
        )
        results["test_flight_MAE_mean"]   = flight_mae["flight_MAE"].mean()
        results["test_flight_MAE_median"] = flight_mae["flight_MAE"].median()
        print(f"  [flight]   mean MAE={results['test_flight_MAE_mean']:.4f}  "
              f"median MAE={results['test_flight_MAE_median']:.4f}")

    print(f"  [overall]  "
          f"test MAE={results['test_MAE']:.4f}  "
          f"RMSE={results['test_RMSE']:.4f}  "
          f"R²={results['test_R2']:.4f}")

    return results, model, test_pred


#diagnostic plots

def save_diagnostic_plots(model_name, dataset_name, y_test, test_pred, plot_dir):
    """
    used to save visual checks of model performance. It created predicted-versus-actual plots, residual distribution plots, and residuals-versus-predicted plots.
    """
    safe = f"{dataset_name}__{model_name}".lower().replace(" ", "_").replace("/", "_")
    residuals = y_test.values - test_pred

    # 1. Predicted vs Actual
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(y_test, test_pred, alpha=0.15, s=4, color="steelblue")
    lim_max = max(y_test.max(), test_pred.max()) * 1.05
    ax.plot([0, lim_max], [0, lim_max], "r--", lw=1.5, label="Perfect prediction")
    ax.set_xlabel("Actual Energy (Wh)")
    ax.set_ylabel("Predicted Energy (Wh)")
    ax.set_title(f"Predicted vs Actual\n{model_name} | {dataset_name}")
    ax.legend()
    plt.tight_layout()
    fig.savefig(plot_dir / f"pred_vs_actual__{safe}.png", dpi=150)
    plt.close(fig)

    # 2. Residual distribution
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(residuals, bins=80, color="coral", edgecolor="none", alpha=0.8)
    ax.axvline(0, color="black", lw=1.2, linestyle="--")
    ax.set_xlabel("Residual (Actual − Predicted) Wh")
    ax.set_ylabel("Count")
    ax.set_title(f"Residual Distribution\n{model_name} | {dataset_name}")
    plt.tight_layout()
    fig.savefig(plot_dir / f"residuals__{safe}.png", dpi=150)
    plt.close(fig)

    # 3. Residuals vs Predicted (heteroscedasticity check)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(test_pred, residuals, alpha=0.10, s=4, color="slategray")
    ax.axhline(0, color="red", lw=1.2, linestyle="--")
    ax.set_xlabel("Predicted Energy (Wh)")
    ax.set_ylabel("Residual (Wh)")
    ax.set_title(f"Residuals vs Predicted\n{model_name} | {dataset_name}")
    plt.tight_layout()
    fig.savefig(plot_dir / f"resid_vs_pred__{safe}.png", dpi=150)
    plt.close(fig)

    print(f"  Plots saved → {safe}")


#feature importance

def save_feature_importance(model, model_name, dataset_name, feature_cols, plot_dir, top_n=20):
    """
    used for models that provide feature importance, such as Random Forest and XGBoost. It saved a ranked plot and CSV file showing which features contributed most to the model predictions.
    """
    if not hasattr(model, "feature_importances_"):
        return

    safe = f"{dataset_name}__{model_name}".lower().replace(" ", "_").replace("/", "_")
    imp  = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
    imp_top = imp.head(top_n)

    fig, ax = plt.subplots(figsize=(8, max(4, top_n * 0.35)))
    imp_top[::-1].plot(kind="barh", ax=ax, color="teal")
    ax.set_xlabel("Feature Importance (MDI)")
    ax.set_title(f"Top {top_n} Feature Importances\n{model_name} | {dataset_name}")
    plt.tight_layout()
    fig.savefig(plot_dir / f"feat_imp__{safe}.png", dpi=150)
    plt.close(fig)

    imp.to_csv(plot_dir / f"feat_imp__{safe}.csv", header=["importance"])
    print(f"  Feature importance saved → feat_imp__{safe}")

## 3. Load non-PCA data

In [30]:
train_no_dr = pd.read_csv(NO_DR_DIR / "train_no_dr_imputed.csv")
val_no_dr   = pd.read_csv(NO_DR_DIR / "val_no_dr_imputed.csv")
test_no_dr  = pd.read_csv(NO_DR_DIR / "test_no_dr_imputed.csv")

feature_cols_no_dr = get_feature_cols(train_no_dr, target_col, id_cols)

X_train_no_dr = train_no_dr[feature_cols_no_dr]
X_val_no_dr   = val_no_dr[feature_cols_no_dr]
X_test_no_dr  = test_no_dr[feature_cols_no_dr]

y_train_no_dr = train_no_dr[target_col]
y_val_no_dr   = val_no_dr[target_col]
y_test_no_dr  = test_no_dr[target_col]

print("Non-PCA train shape:", X_train_no_dr.shape)
print("Non-PCA features   :", feature_cols_no_dr)

Non-PCA train shape: (694549, 11)
Non-PCA features   : ['battery_status__cell_count', 'battery_status__current_a', 'battery_status__current_filtered_a', 'battery_status__discharged_mah', 'battery_status__remaining', 'battery_status__scale', 'battery_status__voltage_filtered_v', 'battery_status__voltage_v', 'cumulative_energy_wh', 'power_w', 'time_s']


## 4. Load PCA data

In [31]:
train_pca = pd.read_csv(PCA_DIR / "train_pca_95.csv")
val_pca   = pd.read_csv(PCA_DIR / "val_pca_95.csv")
test_pca  = pd.read_csv(PCA_DIR / "test_pca_95.csv")

feature_cols_pca = get_feature_cols(train_pca, target_col, id_cols)

X_train_pca = train_pca[feature_cols_pca]
X_val_pca   = val_pca[feature_cols_pca]
X_test_pca  = test_pca[feature_cols_pca]

y_train_pca = train_pca[target_col]
y_val_pca   = val_pca[target_col]
y_test_pca  = test_pca[target_col]

print("PCA train shape  :", X_train_pca.shape)
print("PCA feature count:", len(feature_cols_pca))
print("PCA features[:10]:", feature_cols_pca[:10])

PCA train shape  : (694549, 6)
PCA feature count: 6
PCA features[:10]: ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6']


## 5. Naive baseline
Mean predictor naive baseline
This code gives a minimum benchmark. It shows the error we would get if the model did not learn any real pattern and only guessed the average energy value every time.

It does this for both non-PCA dataset and PCA dataset.

It also checks performance separately for normal flights, abnormal flights and each individual flight

it only uses mean of y_train and ignore every input values 

In [33]:
# Mean Predictor baseline
# This model predicts the training target mean for every validation/test row.



def run_mean_baseline(dataset_name, y_train, y_val, y_test, test_df):
    '''
    Calculate one average target value from the training set
    '''
    train_mean = y_train.mean()

    # Predict the same average value for every row
    val_pred = np.full(len(y_val), train_mean)
    test_pred = np.full(len(y_test), train_mean)

    # Overall validation and test metrics
    result = {
        "dataset": dataset_name,
        "model": "Mean Predictor",
        "train_target_mean": train_mean,

        "val_MAE": mean_absolute_error(y_val, val_pred),
        "val_RMSE": rmse(y_val, val_pred),
        "val_R2": r2_score(y_val, val_pred),

        "test_MAE": mean_absolute_error(y_test, test_pred),
        "test_RMSE": rmse(y_test, test_pred),
        "test_R2": r2_score(y_test, test_pred)
    }

    # Normal and abnormal test performance
    for label in ["normal", "abnormal"]:
        mask = test_df["dataset_label"].values == label

        if mask.sum() > 0:
            y_true_label = y_test.values[mask]
            y_pred_label = test_pred[mask]

            result[f"test_MAE_{label}"] = mean_absolute_error(y_true_label, y_pred_label)
            result[f"test_RMSE_{label}"] = rmse(y_true_label, y_pred_label)
            result[f"test_R2_{label}"] = r2_score(y_true_label, y_pred_label)

    # Flight-level MAE without warning messages
    flight_errors = test_df[["log_id"]].copy()
    flight_errors["absolute_error"] = np.abs(y_test.values - test_pred)

    flight_mae = (
        flight_errors
        .groupby("log_id")["absolute_error"]
        .mean()
    )

    result["test_flight_MAE_mean"] = flight_mae.mean()
    result["test_flight_MAE_median"] = flight_mae.median()

    return result


# Run baseline for both datasets
naive_results = []

naive_results.append(
    run_mean_baseline(
        dataset_name="non_pca_features",
        y_train=y_train_no_dr,
        y_val=y_val_no_dr,
        y_test=y_test_no_dr,
        test_df=test_no_dr
    )
)

naive_results.append(
    run_mean_baseline(
        dataset_name="pca_features",
        y_train=y_train_pca,
        y_val=y_val_pca,
        y_test=y_test_pca,
        test_df=test_pca
    )
)

# Clean results table
naive_results_df = pd.DataFrame(naive_results).round(4)

display(naive_results_df)


,dataset,model,train_target_mean,val_MAE,val_RMSE,val_R2,test_MAE,test_RMSE,test_R2,test_MAE_normal,test_RMSE_normal,test_R2_normal,test_MAE_abnormal,test_RMSE_abnormal,test_R2_abnormal,test_flight_MAE_mean,test_flight_MAE_median
0,non_pca_features,Mean Predictor,2.5281,1.7419,2.0664,-0.0623,1.5109,1.9738,-0.1036,1.6283,2.1438,-0.0692,1.2017,1.4327,-0.3932,1.4106,1.3801
1,pca_features,Mean Predictor,2.5281,1.7419,2.0664,-0.0623,1.5109,1.9738,-0.1036,1.6283,2.1438,-0.0692,1.2017,1.4327,-0.3932,1.4106,1.3801


The RMSE is higher than MAE because RMSE gives more penalty to large errors. This means some predictions were much worse than the average error.

R² tells how much of the variation in the target was explained by the model.

R² = 1.0   very good
R² = 0.0   about same as predicting the average
R² < 0     worse than a simple average baseline

In UAV telemetry, energy use changes because of power demand, current draw, flight stage, battery condition, and abnormal behaviour. R² helps check whether a model is actually capturing these changes.

Low or negative R²:
The model is not explaining the changing energy behaviour.

Higher R²:
The model is learning useful patterns from telemetry.

The abnormal MAE is lower, but the abnormal R² is worse. This can happen if abnormal flights have a different target distribution. The fixed average prediction may be closer in absolute value sometimes, but it still does not explain the variation properly.

Accuracy, precision, recall, and F1 are not suitable here because this is a regression problem, not a classification problem.

## It proves that simply predicting the average 60-second energy use is not enough.

## 6. Power Persistence Baseline

In [35]:
# Assumption: current power use continues for the next 60 seconds.
# Energy in Wh = Power in W × time in hours
# For 60 seconds: predicted energy = power_w × (60 / 3600) = power_w / 60

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def run_power_persistence_baseline(dataset_name, val_df, test_df, y_val, y_test):
    
    # Predict next 60-second energy using current power
    val_pred = val_df["power_w"].values * (60 / 3600)
    test_pred = test_df["power_w"].values * (60 / 3600)

    # Avoid negative energy prediction if any negative power values exist
    val_pred = np.maximum(val_pred, 0)
    test_pred = np.maximum(test_pred, 0)

    result = {
        "dataset": dataset_name,
        "model": "Power Persistence Baseline",

        "val_MAE": mean_absolute_error(y_val, val_pred),
        "val_RMSE": rmse(y_val, val_pred),
        "val_R2": r2_score(y_val, val_pred),

        "test_MAE": mean_absolute_error(y_test, test_pred),
        "test_RMSE": rmse(y_test, test_pred),
        "test_R2": r2_score(y_test, test_pred)
    }

    # Normal and abnormal test performance
    for label in ["normal", "abnormal"]:
        mask = test_df["dataset_label"].values == label

        if mask.sum() > 0:
            y_true_label = y_test.values[mask]
            y_pred_label = test_pred[mask]

            result[f"test_MAE_{label}"] = mean_absolute_error(y_true_label, y_pred_label)
            result[f"test_RMSE_{label}"] = rmse(y_true_label, y_pred_label)
            result[f"test_R2_{label}"] = r2_score(y_true_label, y_pred_label)

    # Flight-level MAE
    flight_errors = test_df[["log_id"]].copy()
    flight_errors["absolute_error"] = np.abs(y_test.values - test_pred)

    flight_mae = (
        flight_errors
        .groupby("log_id")["absolute_error"]
        .mean()
    )

    result["test_flight_MAE_mean"] = flight_mae.mean()
    result["test_flight_MAE_median"] = flight_mae.median()

    return result


# Run power persistence baseline only on non-PCA dataset
power_baseline_result = run_power_persistence_baseline(
    dataset_name="non_pca_features",
    val_df=val_no_dr,
    test_df=test_no_dr,
    y_val=y_val_no_dr,
    y_test=y_test_no_dr
)

power_baseline_df = pd.DataFrame([power_baseline_result]).round(4)

display(power_baseline_df)

,dataset,model,val_MAE,val_RMSE,val_R2,test_MAE,test_RMSE,test_R2,test_MAE_normal,test_RMSE_normal,test_R2_normal,test_MAE_abnormal,test_RMSE_abnormal,test_R2_abnormal,test_flight_MAE_mean,test_flight_MAE_median
0,non_pca_features,Power Persistence Baseline,0.3435,0.7545,0.8584,0.3602,0.8609,0.7901,0.3797,0.9463,0.7917,0.309,0.5785,0.7728,0.3174,0.2951


PCA dataset is not used here because the Power Persistence Baseline needs the real power_w column.

## 7.  Train all models with full diagnostics

In [36]:
# Final baseline + model comparison
all_results = []
trained_models = {}

#baseline results

# Mean Predictor baseline results were already created earlier
for row in naive_results:
    row = row.copy()
    row["model_group"] = "baseline"
    row["note"] = "Predicts training target mean for every row"
    all_results.append(row)

# Power Persistence baseline was already created earlier
power_row = power_baseline_result.copy()
power_row["model_group"] = "baseline"
power_row["note"] = "Uses current power_w to estimate next 60-second energy"
all_results.append(power_row)

# dataset setup for trained models
model_datasets = {
    "non_pca_features": {
        "X_train": X_train_no_dr,
        "y_train": y_train_no_dr,
        "X_val": X_val_no_dr,
        "y_val": y_val_no_dr,
        "X_test": X_test_no_dr,
        "y_test": y_test_no_dr,
        "test_df": test_no_dr,
        "feature_cols": feature_cols_no_dr,
        "uses_pca": False
    },

    "pca_features": {
        "X_train": X_train_pca,
        "y_train": y_train_pca,
        "X_val": X_val_pca,
        "y_val": y_val_pca,
        "X_test": X_test_pca,
        "y_test": y_test_pca,
        "test_df": test_pca,
        "feature_cols": feature_cols_pca,
        "uses_pca": True
    }
}

# Define models to train
def get_models():
    models = {
        "Linear Regression": LinearRegression(),

        "Random Forest": RandomForestRegressor(
            n_estimators=200,
            max_depth=None,
            random_state=42,
            n_jobs=-1
        )
    }

    if XGBOOST_AVAILABLE:
        models["XGBoost"] = XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1,
            verbosity=0
        )

    return models


In [42]:
# Core evaluation function without pandas FutureWarning

def evaluate_model(
    model_name,
    dataset_name,
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    test_df=None
):
    """
    Train a model and calculate validation, test, normal/abnormal,
    and flight-level performance metrics.
    """

    print(f"\n{'='*60}")
    print(f"  Training : {model_name}  |  Dataset : {dataset_name}")
    print(f"{'='*60}")

    # Train model
    model.fit(X_train, y_train)

    # Predict validation and test sets
    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)

    # Overall metrics
    results = {
        "dataset": dataset_name,
        "model": model_name,

        "val_MAE": mean_absolute_error(y_val, val_pred),
        "val_RMSE": rmse(y_val, val_pred),
        "val_R2": r2_score(y_val, val_pred),

        "test_MAE": mean_absolute_error(y_test, test_pred),
        "test_RMSE": rmse(y_test, test_pred),
        "test_R2": r2_score(y_test, test_pred),
    }

    # Normal and abnormal test performance
    if test_df is not None:
        for label in ["normal", "abnormal"]:
            mask = test_df["dataset_label"].values == label

            if mask.sum() == 0:
                continue

            y_true_label = y_test.values[mask]
            y_pred_label = test_pred[mask]

            results[f"test_MAE_{label}"] = mean_absolute_error(
                y_true_label,
                y_pred_label
            )

            results[f"test_RMSE_{label}"] = rmse(
                y_true_label,
                y_pred_label
            )

            results[f"test_R2_{label}"] = r2_score(
                y_true_label,
                y_pred_label
            )

            print(
                f"  [{label:8s}]  "
                f"MAE={results[f'test_MAE_{label}']:.4f}  "
                f"RMSE={results[f'test_RMSE_{label}']:.4f}  "
                f"R²={results[f'test_R2_{label}']:.4f}"
            )

    # Flight-level MAE without groupby.apply warning
    if test_df is not None and "log_id" in test_df.columns:
        flight_errors = test_df[["log_id"]].copy()

        flight_errors["absolute_error"] = np.abs(
            y_test.values - test_pred
        )

        flight_mae = (
            flight_errors
            .groupby("log_id")["absolute_error"]
            .mean()
        )

        results["test_flight_MAE_mean"] = flight_mae.mean()
        results["test_flight_MAE_median"] = flight_mae.median()

        print(
            f"  [flight]   "
            f"mean MAE={results['test_flight_MAE_mean']:.4f}  "
            f"median MAE={results['test_flight_MAE_median']:.4f}"
        )

    print(
        f"  [overall]  "
        f"test MAE={results['test_MAE']:.4f}  "
        f"RMSE={results['test_RMSE']:.4f}  "
        f"R²={results['test_R2']:.4f}"
    )

    return results, model, test_pred

In [43]:


simple_table = results_df_sorted.copy()

simple_table = simple_table[
    ~(
        (simple_table["model"] == "Mean Predictor") &
        (simple_table["dataset"] == "pca_features")
    )
].copy()


simple_table["Dataset"] = simple_table["dataset"].replace({
    "non_pca_features": "Non-PCA",
    "pca_features": "PCA"
})

simple_table["Type"] = simple_table["model_group"].replace({
    "baseline": "Baseline",
    "trained_model": "Trained model"
})


simple_table.loc[
    simple_table["model"] == "Mean Predictor",
    "Dataset"
] = "Both"


simple_table = simple_table[
    [
        "Type",
        "Dataset",
        "model",
        "test_MAE",
        "test_RMSE",
        "test_R2",
        "test_flight_MAE_median"
    ]
].copy()


simple_table = simple_table.rename(columns={
    "model": "Model",
    "test_MAE": "Test MAE",
    "test_RMSE": "Test RMSE",
    "test_R2": "Test R²",
    "test_flight_MAE_median": "Flight median MAE"
})


simple_table = simple_table.sort_values("Test RMSE", ascending=True).reset_index(drop=True)


simple_table.insert(0, "Rank", range(1, len(simple_table) + 1))


numeric_cols = ["Test MAE", "Test RMSE", "Test R²", "Flight median MAE"]
simple_table[numeric_cols] = simple_table[numeric_cols].round(3)

display(simple_table)


simple_table.to_csv(
    OUTPUT_DIR / "simplified_model_comparison_table.csv",
    index=False
)

print("Simplified model comparison table saved.")

,Rank,Type,Dataset,Model,Test MAE,Test RMSE,Test R²,Flight median MAE
0,1,Trained model,Non-PCA,Linear Regression,0.321,0.794,0.821,0.215
1,2,Trained model,Non-PCA,XGBoost,0.324,0.852,0.794,0.155
2,3,Baseline,Non-PCA,Power Persistence Baseline,0.360,0.861,0.790,0.295
3,4,Trained model,Non-PCA,Random Forest,0.339,0.865,0.788,0.176
4,5,Trained model,PCA,XGBoost,0.398,0.937,0.751,0.199
5,6,Trained model,PCA,Random Forest,0.414,0.942,0.748,0.261
6,7,Trained model,PCA,Linear Regression,0.609,1.074,0.673,0.426
7,8,Baseline,Both,Mean Predictor,1.511,1.974,-0.104,1.380


Simplified model comparison table saved.


## 9. Save models

In [44]:
for name, model in trained_models.items():
    safe_name = name.lower().replace(" ", "_").replace("/", "_")
    joblib.dump(model, OUTPUT_DIR / f"{safe_name}.joblib")
    print(f"Saved: {safe_name}.joblib")

print("\nAll models saved.")

Saved: non_pca_features__linear_regression.joblib
Saved: non_pca_features__random_forest.joblib
Saved: non_pca_features__xgboost.joblib
Saved: pca_features__linear_regression.joblib
Saved: pca_features__random_forest.joblib
Saved: pca_features__xgboost.joblib

All models saved.


## 10. Summary 

In [46]:

print("FINAL MODEL COMPARISON SUMMARY")
print("="*70)

# Show best model based on test RMSE
best_model = results_df_sorted.iloc[0]

print("\nBest overall model:")
print(f"Model   : {best_model['model']}")
print(f"Dataset : {best_model['dataset']}")
print(f"Test MAE: {best_model['test_MAE']:.4f}")
print(f"Test RMSE: {best_model['test_RMSE']:.4f}")
print(f"Test R² : {best_model['test_R2']:.4f}")

print("\nMain result table:")
display(simple_table)

print("\nFiles saved:")
print(f"Plots   : {PLOT_DIR}")
print(f"Results : {OUTPUT_DIR}")

print("="*70)

FINAL MODEL COMPARISON SUMMARY

Best overall model:
Model   : Linear Regression
Dataset : non_pca_features
Test MAE: 0.3208
Test RMSE: 0.7944
Test R² : 0.8212

Main result table:


,Rank,Type,Dataset,Model,Test MAE,Test RMSE,Test R²,Flight median MAE
0,1,Trained model,Non-PCA,Linear Regression,0.321,0.794,0.821,0.215
1,2,Trained model,Non-PCA,XGBoost,0.324,0.852,0.794,0.155
2,3,Baseline,Non-PCA,Power Persistence Baseline,0.360,0.861,0.790,0.295
3,4,Trained model,Non-PCA,Random Forest,0.339,0.865,0.788,0.176
4,5,Trained model,PCA,XGBoost,0.398,0.937,0.751,0.199
5,6,Trained model,PCA,Random Forest,0.414,0.942,0.748,0.261
6,7,Trained model,PCA,Linear Regression,0.609,1.074,0.673,0.426
7,8,Baseline,Both,Mean Predictor,1.511,1.974,-0.104,1.380



Files saved:
Plots   : /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/baseline_models/plots
Results : /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/baseline_models
